In [10]:
import torch
import torch.nn.functional as F

import pandas as pd

from openai import OpenAI
import numpy as np

import os
from dotenv import load_dotenv

In [12]:
load_dotenv()

EMBED_MODEL_SMALL = "text-embedding-3-small"  # 1536 вимірів
EMBED_MODEL_LARGE = "text-embedding-3-large"  # 3072 вимірів
LLM_MODEL = "gpt-4o-mini"  # для генерації відповідей
API_KEY = os.getenv("API_KEY")

client = OpenAI(api_key=API_KEY)

words = [
    "vision",
    "color",
    "red",
    "orange",
    "yellow",
    "green",
    "blue",
    "violet",
    "purple",
    "lilac",
    "taste",
    "bitter",
    "sweet",
    "sour"
]

def embed(text: str, model: str) -> np.ndarray:
    model_name = f"text-embedding-3-{model}"
    response = client.embeddings.create(model=model_name, input=text)
    return torch.tensor(response.data[0].embedding, dtype=torch.float32)
    # return np.array(response.data[0].embedding)

# cosine similarity
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [13]:
# embeddings_small = torch.zeros(len(words), 1536)
# embeddings_large = torch.zeros(len(words), 3072)
# models = ["small", "large"]

embeddings = {
    "small": torch.zeros(len(words), 1536),
    "large": torch.zeros(len(words), 3072),
}

for m in ["small", "large"]:
    for i in range(len(words)):
        embeddings[m][i] = embed(words[i], m)

In [17]:
# Sigmoid function
# thresholds = [round(x / 7, 2) for x in [0.2, 0.4, 1, 3, 5, 6, 7 ]]
thresholds = {
    "small": [0.065, 0.07, 0.075, 0.08, 0.085],
    "large": [0.05, 0.055, 0.065, 0.07]
}
iteration = "iteration_4"
os.makedirs(f"model_artifacts/{iteration}", exist_ok=True)

# print(thresholds)

try:
    for m in embeddings:
        for t in thresholds[m]:
    
            # filtering
            B = (embeddings[m] >= t).int()
    
            C = B @ B.T
        
            print(f"model_size: {m} -- threshold = {t}, nonzoer(B) = {len((B == 1).nonzero())}")
        
            df = pd.DataFrame(C, index=words, columns=words)
    
            # Create a folder
            os.makedirs(f"model_artifacts/{iteration}/{m}", exist_ok=True)
            df.to_csv(f"model_artifacts/{iteration}/{m}/filtration_1-cov_matrix_{t}.csv", mode='x')
            
except Exception as e:
    # Catch specific exceptions and access the exception object
    print(f"An error occurred: {e}")

model_size: small -- threshold = 0.065, nonzoer(B) = 172
model_size: small -- threshold = 0.07, nonzoer(B) = 116
model_size: small -- threshold = 0.075, nonzoer(B) = 81
model_size: small -- threshold = 0.08, nonzoer(B) = 60
model_size: small -- threshold = 0.085, nonzoer(B) = 43
model_size: large -- threshold = 0.05, nonzoer(B) = 234
model_size: large -- threshold = 0.055, nonzoer(B) = 148
model_size: large -- threshold = 0.065, nonzoer(B) = 51
model_size: large -- threshold = 0.07, nonzoer(B) = 35


In [18]:
# Gauss
deltas = {
    "small": [0.0007, 0.0008, 0.00085, 0.0009, 0.00095],
    "large": [0.0003, 0.0005, 0.0007, 0.00085]
}
# print(deltas)

try:
    for m in embeddings:
        for d in deltas[m]:
            # filtering
            B = (torch.abs(embeddings[m]) <= d).int()
            C = B @ B.T
        
            print(f"model_size: {m} -- delta = {d}, nonzoer(B) = {len((B == 1).nonzero())}")
        
            df = pd.DataFrame(C, index=words, columns=words)
            df.to_csv(f"model_artifacts/{iteration}/{m}/filtration_2-cov_matrix_{d}.csv", mode='x')

except Exception as e:
    # Catch specific exceptions and access the exception object
    print(f"An error occurred: {e}")

model_size: small -- delta = 0.0007, nonzoer(B) = 490
model_size: small -- delta = 0.0008, nonzoer(B) = 549
model_size: small -- delta = 0.00085, nonzoer(B) = 580
model_size: small -- delta = 0.0009, nonzoer(B) = 619
model_size: small -- delta = 0.00095, nonzoer(B) = 661
model_size: large -- delta = 0.0003, nonzoer(B) = 678
model_size: large -- delta = 0.0005, nonzoer(B) = 1097
model_size: large -- delta = 0.0007, nonzoer(B) = 1504
model_size: large -- delta = 0.00085, nonzoer(B) = 1814
